# Phase 4 — Aggregate and analyze

**Goal.** Pull every per-chunk result back together, look at the dataset in aggregate, and decide which molecules pass your stability filter.

**What "stability" means here.** Two heuristics combined:

1. `energy < --energy-max` — the relaxed geometry converged to something chemically sensible (non-physical SCF divergences often leave huge positive energies).
2. `gap > --gap-min` — the HOMO/LUMO gap is large enough that the molecule isn't trivially reactive. Small-gap compounds are often colored, often open-shell, often photoreactive — usually not what a discovery screen is trying to surface.

These are *triage* filters, not final chemistry verdicts. The real question after this notebook is which surviving molecules to send for more expensive follow-up (larger basis sets, excited-state calculations, synthesis planning).

**Where to run this.** Head node, plain `python3.12` kernel with pandas + matplotlib. No GPU needed.

**Prerequisite.** Phase 2's array has finished and Phase 4 (the Slurm-dependency-triggered job) has already written `data/results/stable.csv` + `logs/phase4_summary.txt`.

## 1. Load every per-chunk result into one DataFrame

We glob `data/results/result_*.csv` rather than reading `stable.csv` because we want to see the **unfiltered** distribution — the pre-filter view is what tells you whether the thresholds are in the right place.

In [ ]:
import glob
import pandas as pd

paths = sorted(glob.glob('data/results/result_*.csv'))
df = pd.concat((pd.read_csv(p) for p in paths), ignore_index=True)
print(f'loaded {len(paths)} chunk files, {len(df)} molecules total')
df.head()

## 2. Summary statistics

Quick gut-check. Look for:

- **Implausible `energy` outliers** (e.g., one molecule with energy orders-of-magnitude different from the rest) → something went wrong in that worker.
- **Very negative `homo` or very positive `lumo`** → unusual electronic structure, possibly interesting but also possibly numeric trouble.
- **`gap` = 0 or negative** → something's broken; the LUMO should always be above the HOMO.
- **`n_atoms`** distribution — reassure yourself the screen hit the molecule-size range you expected.

In [ ]:
df[['energy', 'homo', 'lumo', 'gap', 'n_atoms']].describe()

## 3. Distribution of HOMO/LUMO gaps

**Why plot before filtering?** The stability filter is only useful if the threshold lands in a meaningful spot on the distribution. A histogram tells you whether `gap_min = 1 eV` is aggressive (chops off half the dataset) or conservative (lets almost everything through).

The red dashed line marks the default `gap_min = 1 eV`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['gap'], bins=20, edgecolor='black')
ax.set_xlabel('HOMO-LUMO gap (eV)')
ax.set_ylabel('molecules')
ax.axvline(1.0, color='red', linestyle='--', label='default gap_min = 1 eV')
ax.legend()
plt.tight_layout()

### What to do if the plot looks weird

- **All molecules in a narrow band** — fine for a tiny smoke-test run. Scale up Phase 1.
- **Bimodal distribution** — often means two structurally distinct groups in your input (e.g., aromatics vs aliphatics). Consider filtering pre-screening or splitting the analysis.
- **Long tail of very small gaps** — small-gap species can be interesting (dyes, photocatalysts) but also tend to be the ones where single-reference DFT is unreliable. Revisit with TD-DFT or CASSCF if those are your targets.


## 4. Apply the stability filter

Same two thresholds that `4_aggregate_and_learn.py` applied in production. Change them here to explore; when you're happy, edit the Phase 4 CLI defaults (`--energy-max`, `--gap-min`) so future runs write the filter you actually want.

In [ ]:
energy_max = 0.0   # Hartree
gap_min    = 1.0   # eV

stable = df[(df['energy'] < energy_max) & (df['gap'] > gap_min)].copy()
print(f'{len(stable)}/{len(df)} molecules pass (energy < {energy_max}, gap > {gap_min} eV)')
stable.sort_values('gap', ascending=False).head(10)

## 5. Sanity-check against the Slurm-produced output

`4_aggregate_and_learn.py` already wrote `data/results/stable.csv` when the array finished, using the defaults. Confirm that our interactive filter agrees with it — if the counts match, we're consistent.

In [ ]:
slurm_stable = pd.read_csv('data/results/stable.csv')
print(f'Slurm wrote {len(slurm_stable)} rows; notebook filter gives {len(stable)} rows')
assert len(slurm_stable) == len(stable), 'mismatch — thresholds differ from what Phase 4 ran with'

### Production summary

One line of provenance so you can later tell which thresholds produced `stable.csv`.

In [ ]:
!cat logs/phase4_summary.txt

## 6. Next step — close the active-learning loop

The `retrain_surrogate(df)` function at the bottom of `4_aggregate_and_learn.py` is a stub. The intended flow:

1. These freshly-labeled "stable" rows become training data for a fast surrogate model (graph NN, Tanimoto-kNN, Gaussian process on molecular fingerprints...).
2. The surrogate ranks the next batch of GDB-17 candidates.
3. Feed those top-ranked candidates back into Phase 1 as a filtered input — the screen gets steered toward interesting chemistry instead of scanning GDB-17 uniformly.

That's the machine-learning-in-the-loop piece. Everything up to this notebook is the ground-truth generator that feeds it.